### Evaluation of the SwarmMate project.

Ragas’ current docs support evaluating a RAG app with evaluate() over a dataset, and the stable metric catalog includes Context Precision, Context Recall, Answer Relevancy, Faithfulness, Semantic Similarity, and rubric-based scoring. The docs also support synthetic testset generation, but for your course requirement the simplest reliable path is to start with a manual synthetic set that you author from your sample workpapers. 

For the metrics here is how I would map the metrics:

1. citation relevance / retrieval relevance → ContextPrecision

2. answer completeness → AnswerRelevancy

3. groundedness to source docs → Faithfulness

4. artifact usefulness / completeness for audit planning → DomainSpecificRubrics with your own audit-specific rubric. 

Below, the eval_cases cell is your synthetic set, and the rest runs your existing SwarmMate graph, builds a Ragas dataset, scores it, and gives you a form-ready summary.

1) Install notebook dependencies

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path(r"C:\MyWorkspace\Assignments\Krishanu_AIE9_Assignments\swarmmate_audit_workpaper_adapter\swarmmate_audit_workpaper_adapter")
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env", override=True)

True

In [2]:
!python -m pip install -U "ragas>=0.3.5" datasets openai pandas python-dotenv nest_asyncio

2) Load your project and environment

In [3]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()

# Walk upward until we find your repo root
while not (PROJECT_ROOT / "pyproject.toml").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

print("PROJECT_ROOT =", PROJECT_ROOT)

sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env", override=True)

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is missing from .env"

# Keep your existing defaults unless you intentionally override them later
os.environ.setdefault("RAG_DATA_DIR", str(PROJECT_ROOT / "data"))
os.environ.setdefault("RAG_TOP_K", "4")
os.environ.setdefault("RAG_CHUNK_SIZE", "900")
os.environ.setdefault("RAG_CHUNK_OVERLAP", "150")

PROJECT_ROOT = c:\MyWorkspace\Assignments\Krishanu_AIE9_Assignments\swarmmate_audit_workpaper_adapter\swarmmate_audit_workpaper_adapter


'150'

3) Import SwarmMate + Ragas

In [4]:
from openai import OpenAI
from datasets import Dataset
from langchain_core.messages import HumanMessage

from ragas import evaluate
from ragas.llms import llm_factory
from ragas.embeddings import OpenAIEmbeddings
from ragas import evaluate
from ragas.metrics import (
    LLMContextPrecisionWithReference,
    LLMContextRecall,
    ResponseRelevancy,
)



from app.graphs.workpaper_adapter_with_guardrails import graph
from app import rag as rag_module


c:\venvs\swarmmate_eval\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\SinhaK\AppData\Local\Temp\ipykernel_36852\863770522.py:9: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference
  from ragas.metrics import (
C:\Users\SinhaK\AppData\Local\Temp\ipykernel_36852\863770522.py:9: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import (
C:\Users\SinhaK\AppData\Local\Temp\ipykernel_36852\863770522.py:9: DeprecationWar

4) Set up the Ragas judge LLM + embeddings

In [5]:
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

RAGAS_EVAL_MODEL = os.getenv("RAGAS_EVAL_MODEL", "gpt-4.1-mini")
RAGAS_EVAL_EMBED_MODEL = os.getenv("RAGAS_EVAL_EMBED_MODEL", "text-embedding-3-small")

judge_llm = llm_factory(RAGAS_EVAL_MODEL, client=client)
judge_embeddings = OpenAIEmbeddings(client=client, model=RAGAS_EVAL_EMBED_MODEL)

print("Judge model:", RAGAS_EVAL_MODEL)
print("Judge embedding model:", RAGAS_EVAL_EMBED_MODEL)

Judge model: gpt-4.1-mini
Judge embedding model: text-embedding-3-small


5) Create your synthetic evaluation set

This is the easiest, most controllable way to satisfy the “synthetic set + baseline score” requirement.

In [6]:
eval_cases = [
    {
        "case_id": "open_items",
        "user_input": "What items are still outstanding from the client? Answer in bullets with citations.",
        "reference": (
            "Outstanding items are pricing override support for the top 10 custom contracts, "
            "year-end bank reconciliation approval evidence, an updated lease schedule for the new warehouse, "
            "and detailed support for the inventory obsolescence reserve."
        ),
    },
    {
        "case_id": "client_background",
        "user_input": "Summarize the client background for audit planning with citations.",
        "reference": (
            "Lakeview Manufacturing is an FY2025 audit engagement in industrial components manufacturing. "
            "The client uses an on-prem accounting package plus manual spreadsheet support files, has a "
            "2025-12-31 year end, sells mainly domestic wholesale contracts plus a smaller custom-order "
            "service line, improved gross margin in Q4 after a pricing update, and lost a senior accountant "
            "in October with duties redistributed."
        ),
    },
    {
        "case_id": "control_risks",
        "user_input": "What control and audit risk areas should the team focus on? Use bullets and cite the sources.",
        "reference": (
            "Key risks are delayed review of November and December bank reconciliations, one reconciling item "
            "older than 45 days at year end, pricing override support stored in email instead of a controlled "
            "workflow, manual spreadsheet use for the inventory obsolescence reserve update, prior-year pressure "
            "around manual journal entries near period end, and staffing changes after a senior accountant left."
        ),
    },
    {
        "case_id": "mje_reserve",
        "user_input": "Why is the manual journal entry reserve a likely follow-up area for the audit team? Cite your answer.",
        "reference": (
            "It is a likely follow-up area because prior-year pressure points included manual journal entries near "
            "period end, and the trial balance shows the Manual Journal Entry Reserve increased sharply from 20000 "
            "in fiscal 2024 to 185000 in fiscal 2025."
        ),
    },
    {
        "case_id": "audit_planning_memo",
        "user_input": (
            "Draft a short audit planning memo for this client. Highlight likely risk areas, "
            "open requests, and include citations."
        ),
        "reference": (
            "The audit planning memo should cover Lakeview Manufacturing's FY2025 background, note risks around "
            "delayed bank reconciliation review, stale reconciling items, pricing override support held in email, "
            "manual spreadsheet-based obsolescence reserve work, manual journal entries, and staffing changes, and "
            "list the outstanding client requests: pricing override support, bank reconciliation approval evidence, "
            "updated lease schedule, and inventory reserve support."
        ),
    },
]

pd.DataFrame(eval_cases)[["case_id", "user_input"]]


,case_id,user_input
0,open_items,What items are still outstanding from the clie...
1,client_background,Summarize the client background for audit plan...
2,control_risks,What control and audit risk areas should the t...
3,mje_reserve,Why is the manual journal entry reserve a like...
4,audit_planning_memo,Draft a short audit planning memo for this cli...


6) Create helper functions to run your current SwarmMate graph

In [7]:
def reset_retriever(*, top_k=4, chunk_size=900, chunk_overlap=150, data_dir=None):
    os.environ["RAG_TOP_K"] = str(top_k)
    os.environ["RAG_CHUNK_SIZE"] = str(chunk_size)
    os.environ["RAG_CHUNK_OVERLAP"] = str(chunk_overlap)

    if data_dir is not None:
        os.environ["RAG_DATA_DIR"] = str(data_dir)

    # Clear cached in-memory vector store so the new settings take effect
    if hasattr(rag_module, "_get_vector_store"):
        rag_module._get_vector_store.cache_clear()


def run_swarmmate(user_input: str) -> dict:
    result = graph.invoke(
        {
            "messages": [
                HumanMessage(content=user_input)
            ]
        }
    )

    response = (result.get("artifact_markdown") or result.get("final_response") or "").strip()
    evidence = result.get("evidence", []) or []

    retrieved_contexts = [item["excerpt"] for item in evidence]
    retrieved_sources = [f'{item["citation"]}:{item["source"]}' for item in evidence]

    return {
        "response": response,
        "retrieved_contexts": retrieved_contexts,
        "retrieved_sources": retrieved_sources,
        "num_contexts": len(retrieved_contexts),
        "raw_result": result,
    }


def build_run_df(eval_cases, *, top_k=4, chunk_size=900, chunk_overlap=150, data_dir=None):
    reset_retriever(
        top_k=top_k,
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        data_dir=data_dir,
    )

    rows = []
    for case in eval_cases:
        out = run_swarmmate(case["user_input"])
        rows.append(
            {
                **case,
                "response": out["response"],
                "retrieved_contexts": out["retrieved_contexts"],
                "retrieved_sources_text": ", ".join(out["retrieved_sources"]),
                "num_contexts": out["num_contexts"],
            }
        )
    return pd.DataFrame(rows)



7) Define the Ragas metrics

DomainSpecificRubrics is the easiest way to turn your screenshot’s “artifact completeness” and “artifact usefulness for audit planning” into scored metrics. 

In [8]:
completeness_rubric = {
    "score1_description": "The response misses most required facts from the reference and fails the task.",
    "score2_description": "The response covers some correct facts but has major omissions that make it incomplete.",
    "score3_description": "The response is mostly correct but misses important details or requested items.",
    "score4_description": "The response is strong and mostly complete with only minor omissions.",
    "score5_description": "The response is fully complete, covers all major requested facts, and matches the reference very well.",
}

usefulness_rubric = {
    "score1_description": "The artifact would not be useful for audit planning or reviewer handoff.",
    "score2_description": "The artifact has limited practical value because it is poorly structured or misses key risk/open-item information.",
    "score3_description": "The artifact is somewhat useful but would still need noticeable reviewer cleanup.",
    "score4_description": "The artifact is useful for audit planning with only minor cleanup needed.",
    "score5_description": "The artifact is highly useful, clearly structured, review-ready, and directly supports audit planning.",
}

ragas_metrics = [
    LLMContextPrecisionWithReference(llm=judge_llm),
    LLMContextRecall(llm=judge_llm),
    ResponseRelevancy(llm=judge_llm, embeddings=judge_embeddings),
]

[type(m).__name__ for m in ragas_metrics]


['LLMContextPrecisionWithReference', 'LLMContextRecall', 'ResponseRelevancy']

8) Run the baseline experiment

In [9]:
baseline_runs_df = build_run_df(
    eval_cases,
    top_k=int(os.getenv("RAG_TOP_K", "4")),
    chunk_size=int(os.getenv("RAG_CHUNK_SIZE", "900")),
    chunk_overlap=int(os.getenv("RAG_CHUNK_OVERLAP", "150")),
)

baseline_runs_df[["case_id", "num_contexts", "retrieved_sources_text"]]


,case_id,num_contexts,retrieved_sources_text
0,open_items,5,"S1:pbc_request_list.md, S2:trial_balance.csv, ..."
1,client_background,6,"S1:client_overview.md, S2:pbc_request_list.md,..."
2,control_risks,6,"S1:control_notes.md, S2:trial_balance.csv, S3:..."
3,mje_reserve,6,"S1:trial_balance.csv, S2:control_notes.md, S3:..."
4,audit_planning_memo,6,"S1:control_notes.md, S2:trial_balance.csv, S3:..."


9) Convert to a Ragas-compatible dataset and evaluate

Ragas’ evaluate() accepts a dataset and returns an evaluation result object with per-metric scores; its docs also show converting results to pandas for analysis. 

In [10]:
from openai import OpenAI
from ragas.llms import llm_factory
from langchain_openai import OpenAIEmbeddings as LCOpenAIEmbeddings
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    timeout=120.0,
    max_retries=5,
)

RAGAS_EVAL_MODEL = os.getenv("RAGAS_EVAL_MODEL", "gpt-4.1")
RAGAS_EVAL_EMBED_MODEL = os.getenv("RAGAS_EVAL_EMBED_MODEL", "text-embedding-3-small")

judge_llm = llm_factory(
    RAGAS_EVAL_MODEL,
    client=client,
    temperature=0,
    max_tokens=4096,
)

judge_embeddings = LangchainEmbeddingsWrapper(
    LCOpenAIEmbeddings(
        model=RAGAS_EVAL_EMBED_MODEL,
        api_key=os.environ["OPENAI_API_KEY"],
    )
)

run_config = RunConfig(
    max_workers=1,
    timeout=180,
    max_retries=5,
)

C:\Users\SinhaK\AppData\Local\Temp\ipykernel_36852\2046600082.py:23: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  judge_embeddings = LangchainEmbeddingsWrapper(


In [11]:
def trim_text(text: str, max_chars: int = 3500) -> str:
    text = (text or "").strip()
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + "\n...[truncated for evaluation]"


def trim_contexts(contexts: list[str], max_contexts: int = 4, max_chars_per_context: int = 1200) -> list[str]:
    trimmed = []
    for ctx in (contexts or [])[:max_contexts]:
        ctx = (ctx or "").strip()
        if len(ctx) > max_chars_per_context:
            ctx = ctx[:max_chars_per_context] + "\n...[truncated for evaluation]"
        trimmed.append(ctx)
    return trimmed

In [12]:
baseline_eval_df = baseline_runs_df.copy()

baseline_eval_df["response_eval"] = baseline_eval_df["response"].apply(lambda x: trim_text(x, max_chars=3500))
baseline_eval_df["retrieved_contexts_eval"] = baseline_eval_df["retrieved_contexts"].apply(
    lambda x: trim_contexts(x, max_contexts=4, max_chars_per_context=1200)
)
baseline_eval_df["reference_eval"] = baseline_eval_df["reference"].apply(lambda x: trim_text(x, max_chars=2500))

baseline_hf_dataset = Dataset.from_pandas(
    baseline_eval_df[
        ["user_input", "retrieved_contexts_eval", "response_eval", "reference_eval"]
    ].rename(
        columns={
            "retrieved_contexts_eval": "retrieved_contexts",
            "response_eval": "response",
            "reference_eval": "reference",
        }
    ),
    preserve_index=False,
)

In [13]:
baseline_result = evaluate(
    dataset=baseline_hf_dataset,
    metrics=ragas_metrics,
    run_config=run_config,
    raise_exceptions=False,
    show_progress=True,
)

baseline_metric_df = baseline_result.to_pandas()
baseline_metric_df

Evaluating:   0%|          | 0/15 [00:00<?, ?it/s]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[2]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[5]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[8]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[11]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[14]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
Evaluating: 100%|██████████| 15/1

,user_input,retrieved_contexts,response,reference,llm_context_precision_with_reference,context_recall,answer_relevancy
0,What items are still outstanding from the clie...,[# PBC request list / open items Received: - F...,# Outstanding Items from Client\n\n- Pricing o...,Outstanding items are pricing override support...,1.000000,1.0,NaN
1,Summarize the client background for audit plan...,[# Client overview - Lakeview Manufacturing - ...,# Client Background Summary for Audit Planning...,Lakeview Manufacturing is an FY2025 audit enga...,1.000000,1.0,NaN
2,What control and audit risk areas should the t...,[# Control notes ## Cash and bank reconciliati...,# Summary of Control Environment\n- Lakeview M...,Key risks are delayed review of November and D...,0.805556,1.0,NaN
3,Why is the manual journal entry reserve a like...,[account: 5990 description: Manual Journal Ent...,# Introduction\nThis summary explains why the ...,It is a likely follow-up area because prior-ye...,0.750000,1.0,NaN
4,Draft a short audit planning memo for this cli...,[# Control notes ## Cash and bank reconciliati...,# Audit Planning Memo\n\n## Introduction and C...,The audit planning memo should cover Lakeview ...,0.805556,1.0,NaN


10) Making sure the notebook reruns the retriever cleanly

In [14]:
def set_retrieval_mode(mode: str, *, fetch_k: int = 12, lambda_mult: float = 0.35):
    os.environ["RAG_RETRIEVAL_MODE"] = mode
    os.environ["RAG_MMR_FETCH_K"] = str(fetch_k)
    os.environ["RAG_MMR_LAMBDA"] = str(lambda_mult)

    if hasattr(rag_module, "_get_vector_store"):
        rag_module._get_vector_store.cache_clear()



In [16]:
set_retrieval_mode("similarity", fetch_k=12, lambda_mult=0.35)

baseline_runs_df = build_run_df(
    eval_cases,
    top_k=int(os.getenv("RAG_TOP_K", "4")),
    chunk_size=int(os.getenv("RAG_CHUNK_SIZE", "900")),
    chunk_overlap=int(os.getenv("RAG_CHUNK_OVERLAP", "150")),
)

baseline_eval_df = baseline_runs_df.copy()
baseline_eval_df["response_eval"] = baseline_eval_df["response"].apply(lambda x: trim_text(x, max_chars=3500))
baseline_eval_df["retrieved_contexts_eval"] = baseline_eval_df["retrieved_contexts"].apply(
    lambda x: trim_contexts(x, max_contexts=4, max_chars_per_context=1200)
)
baseline_eval_df["reference_eval"] = baseline_eval_df["reference"].apply(lambda x: trim_text(x, max_chars=2500))

baseline_hf_dataset = Dataset.from_pandas(
    baseline_eval_df[
        ["user_input", "retrieved_contexts_eval", "response_eval", "reference_eval"]
    ].rename(
        columns={
            "retrieved_contexts_eval": "retrieved_contexts",
            "response_eval": "response",
            "reference_eval": "reference",
        }
    ),
    preserve_index=False,
)

baseline_result = evaluate(
    dataset=baseline_hf_dataset,
    metrics=ragas_metrics,
    run_config=run_config,
    raise_exceptions=False,
    show_progress=True,
)

baseline_metric_df = baseline_result.to_pandas()
baseline_metric_df

Evaluating:   0%|          | 0/15 [00:00<?, ?it/s]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[2]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[5]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[8]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[11]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[14]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
Evaluating: 100%|██████████| 15/1

,user_input,retrieved_contexts,response,reference,llm_context_precision_with_reference,context_recall,answer_relevancy
0,What items are still outstanding from the clie...,[# PBC request list / open items Received: - F...,# Outstanding Items from Client\n\n- Pricing o...,Outstanding items are pricing override support...,1.0,1.0,NaN
1,Summarize the client background for audit plan...,[# Client overview - Lakeview Manufacturing - ...,# Client Background Summary for Audit Planning...,Lakeview Manufacturing is an FY2025 audit enga...,1.0,1.0,NaN
2,What control and audit risk areas should the t...,[# Control notes ## Cash and bank reconciliati...,# Summary of Control Environment\n- Lakeview M...,Key risks are delayed review of November and D...,1.0,1.0,NaN
3,Why is the manual journal entry reserve a like...,[account: 5990 description: Manual Journal Ent...,## Summary of Manual Journal Entry Reserve\n- ...,It is a likely follow-up area because prior-ye...,1.0,1.0,NaN
4,Draft a short audit planning memo for this cli...,[# Control notes ## Cash and bank reconciliati...,# Audit Planning Memo\n\n## Introduction and C...,The audit planning memo should cover Lakeview ...,1.0,0.0,NaN


11) RETRIEVAL UPGRADE USING MMR:

In [19]:
set_retrieval_mode("mmr", fetch_k=12, lambda_mult=0.35)

mmr_runs_df = build_run_df(
    eval_cases,
    top_k=int(os.getenv("RAG_TOP_K", "4")),
    chunk_size=int(os.getenv("RAG_CHUNK_SIZE", "900")),
    chunk_overlap=int(os.getenv("RAG_CHUNK_OVERLAP", "150")),
)

mmr_eval_df = mmr_runs_df.copy()
mmr_eval_df["response_eval"] = mmr_eval_df["response"].apply(lambda x: trim_text(x, max_chars=3500))
mmr_eval_df["retrieved_contexts_eval"] = mmr_eval_df["retrieved_contexts"].apply(
    lambda x: trim_contexts(x, max_contexts=4, max_chars_per_context=1200)
)
mmr_eval_df["reference_eval"] = mmr_eval_df["reference"].apply(lambda x: trim_text(x, max_chars=2500))

mmr_hf_dataset = Dataset.from_pandas(
    mmr_eval_df[
        ["user_input", "retrieved_contexts_eval", "response_eval", "reference_eval"]
    ].rename(
        columns={
            "retrieved_contexts_eval": "retrieved_contexts",
            "response_eval": "response",
            "reference_eval": "reference",
        }
    ),
    preserve_index=False,
)

mmr_result = evaluate(
    dataset=mmr_hf_dataset,
    metrics=ragas_metrics,
    run_config=run_config,
    raise_exceptions=False,
    show_progress=True,
)

mmr_metric_df = mmr_result.to_pandas()
mmr_metric_df

Evaluating:   0%|          | 0/15 [00:00<?, ?it/s]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[2]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[5]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[8]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[11]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[14]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
Evaluating: 100%|██████████| 15/1

,user_input,retrieved_contexts,response,reference,llm_context_precision_with_reference,context_recall,answer_relevancy
0,What items are still outstanding from the clie...,[# PBC request list / open items Received: - F...,# Outstanding Items from Client with Citations...,Outstanding items are pricing override support...,1.000000,1.0,NaN
1,Summarize the client background for audit plan...,[# Client overview - Lakeview Manufacturing - ...,# Client Background Summary for Audit Planning...,Lakeview Manufacturing is an FY2025 audit enga...,1.000000,1.0,NaN
2,What control and audit risk areas should the t...,[# Control notes ## Cash and bank reconciliati...,# Summary of Control Risk Areas\n\n- **Cash an...,Key risks are delayed review of November and D...,0.805556,1.0,NaN
3,Why is the manual journal entry reserve a like...,[account: 5990 description: Manual Journal Ent...,# Introduction\nThe manual journal entry reser...,It is a likely follow-up area because prior-ye...,1.000000,1.0,NaN
4,Draft a short audit planning memo for this cli...,[# Control notes ## Cash and bank reconciliati...,# Audit Planning Memo\n\n## Introduction and C...,The audit planning memo should cover Lakeview ...,0.805556,1.0,NaN


### Comparison table

In [20]:
baseline_summary = {
    "config": "baseline_similarity",
    "context_precision": baseline_metric_df["llm_context_precision_with_reference"].mean(),
    "context_recall": baseline_metric_df["context_recall"].mean(),
}

mmr_summary = {
    "config": "mmr_upgrade",
    "context_precision": mmr_metric_df["llm_context_precision_with_reference"].mean(),
    "context_recall": mmr_metric_df["context_recall"].mean(),
}

comparison_df = pd.DataFrame([baseline_summary, mmr_summary])
comparison_df["delta_precision_vs_baseline"] = comparison_df["context_precision"] - comparison_df.iloc[0]["context_precision"]
comparison_df["delta_recall_vs_baseline"] = comparison_df["context_recall"] - comparison_df.iloc[0]["context_recall"]
comparison_df.round(3)

,config,context_precision,context_recall,delta_precision_vs_baseline,delta_recall_vs_baseline
0,baseline_similarity,1.000,0.8,0.000,0.0
1,mmr_upgrade,0.922,1.0,-0.078,0.2
